# Lab 2: Data Management and Row Selection

## Learning Outcomes

By the end of this lab you will be able to:

- Create analytical subsets of data
- Apply row filtering using logical conditions
- Investigate missing data
- Replace invalid codes with NaN values
- Understand skip patterns in survey datasets
- Recode missing values into meaningful categories
- Use pandas to support data management decisions

This lab builds directly on Lab 1 and continues using the NESARC dataset.

# Part 1: Why Data Management Matters

Before conducting analysis or building predictive models, analysts must understand the quality and meaning of their data.

Data management involves making justified decisions about:

- Which observations should be included
- Which observations should be excluded
- How missing values should be handled
- How coded survey responses should be interpreted

Poor data management often leads to misleading analysis results.

## Load and Prepare the Dataset

In [ ]:
import pandas as pd
import numpy as np

nesarc_data = pd.read_csv('nesarc_pds.csv', low_memory=False)
nesarc_data.columns = map(str.upper, nesarc_data.columns)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x:'%f' % x)

print('Rows:', len(nesarc_data))
print('Columns:', len(nesarc_data.columns))

In [ ]:
nesarc_data['CHECK321'] = nesarc_data['CHECK321'].replace(' ', np.nan)

nesarc_data['TAB12MDX'] = pd.to_numeric(nesarc_data['TAB12MDX'], errors='coerce')
nesarc_data['CHECK321'] = pd.to_numeric(nesarc_data['CHECK321'], errors='coerce')
nesarc_data['S3AQ3C1'] = pd.to_numeric(nesarc_data['S3AQ3C1'], errors='coerce')

# Part 2: Row Selection and Data Subsetting

In Lab 1 frequency distributions were calculated on the entire dataset.

We will now focus on a subset:

- Age between 18 and 25 inclusive
- Smoked within the last 12 months (CHECK321 = 1)

This produces a dataset tailored to the research question.

## Population vs Analytical Sample
The original NESARC dataset contains the full population of respondents. Analysts often create a smaller analytical sample that is relevant to a research question. In this lab we focus on smokers aged 18–25.


In [ ]:
## Logical Operators

== equal to
!= not equal to
> greater than
< less than
>= greater than or equal to
<= less than or equal to
& AND
| OR


In [ ]:
# Example logical expressions
# age >= 18
# age <= 25
# CHECK321 == 1


## Creating Copies of DataFrames
A copy allows experimentation without changing the original dataframe.


In [ ]:
subset2 = nesarc_data.copy()
print('Rows in subset2:', len(subset2))


## Using loc
loc allows row and column selection and updating based on conditions.


In [ ]:
# Simple loc example
# students.loc[students['Age'] > 20]


In [ ]:
print(nesarc_data['AGE'].value_counts(sort=False))


In [ ]:
import matplotlib.pyplot as plt
nesarc_data['AGE'].hist()
plt.title('Age Distribution')
plt.show()


In [ ]:
print('Number of observations before filtering')
print(len(nesarc_data))

In [ ]:
nesarc_data = nesarc_data[
    (nesarc_data['AGE'] >= 18) &
    (nesarc_data['AGE'] <= 25) &
    (nesarc_data['CHECK321'] == 1)
]

print('Number of observations in subset')
print(len(nesarc_data))

In [ ]:
# Exercise
# Verify the subset by generating frequency counts for CHECK321
# There should only be recent smokers in the subset.

In [ ]:
# Exercise
# Generate frequency counts for AGE
# What age occurs most frequently in the subset?

In [ ]:
# Challenge
# Create a copy of the filtered dataset called subset2
# and display the number of observations.

# Part 3: Missing Data Management

A common task in analytics is deciding how to treat unknown or invalid values.

For variable S3AQ3B1:

- Values 1–8 are valid categories.
- Value 9 means unknown.

Unknown values do not help answer the research question and should be converted to missing values (NaN).

In [ ]:
nesarc_data['S3AQ3B1'].dtype

In [ ]:
nesarc_data['S3AQ3B1'] = pd.to_numeric(nesarc_data['S3AQ3B1'])

In [ ]:
print('Counts for S3AQ3B1')
print(nesarc_data['S3AQ3B1'].value_counts(sort=True))

## What is NaN?
NaN means Not a Number and is used by pandas to represent missing values. In the next line of code we replace any NaN's with the number 9.

In [ ]:
nesarc_data['S3AQ3B1'].replace(to_replace=9, value=np.nan, inplace=True)

In [ ]:
print((nesarc_data['S3AQ3B1'] == 9).sum())

In [ ]:
print(nesarc_data['S3AQ3B1'].isnull().sum())

In [ ]:
# Exercise
# Re-run the frequency distribution for S3AQ3B1
# and verify that 9 no longer appears.

# Part 4: Recovering Meaningful Missing Data

Survey datasets frequently contain skip patterns.

Example:

If a person did not drink alcohol in the last 12 months,
there is no reason to ask how often they drank.

In these situations, blank values may actually contain useful information.

In [ ]:
print('count of nulls')
print((nesarc_data['S2AQ8A'].isnull()).sum())

print('count of empty strings')
print((nesarc_data['S2AQ8A'] == '').sum())

print('count of spaces')
print((nesarc_data['S2AQ8A'] == ' ').sum())

In [ ]:
nesarc_data['S2AQ8A'] = nesarc_data['S2AQ8A'].replace(' ', np.nan)

In [ ]:
print('Counts for S2AQ8A')
print(nesarc_data['S2AQ8A'].value_counts(sort=True, dropna=False))

In [ ]:
nesarc_data.loc[
    (nesarc_data['S2AQ3'] != 9) &
    (nesarc_data['S2AQ8A'].isnull()),
    'S2AQ8A'
] = 11

In [ ]:
# Exercise
# Produce a frequency table for S2AQ8A.
# Confirm that category 11 now contains the recovered observations.

# Reflection Questions

1. Why is row selection important?
2. Why are unknown values often replaced with NaN?
3. What is a skip pattern?
4. Why should data management decisions be documented?

## Part 5: Recoding Responses

Sometimes response codes are technically correct but difficult to interpret. Analysts often recode values to make them more intuitive while preserving the meaning of the data. In this section you will reverse-code smoking frequency and create a quantitative approximation of smoking days per month.


In [ ]:
recode1 = {
    1:6,
    2:5,
    3:4,
    4:3,
    5:2,
    6:1
}

nesarc_data['USFREQ'] = nesarc_data['S3AQ3B1'].map(recode1)

nesarc_data['USFREQ'].value_counts(dropna=False)


The new variable USFREQ is easier to interpret because higher values now correspond to more frequent smoking behaviour. This is often useful when creating visualisations and performing statistical analysis.


In [ ]:
recode2 = {
    1:30,
    2:22,
    3:14,
    4:5,
    5:2.5,
    6:1
}

nesarc_data['USFREQMO'] = nesarc_data['S3AQ3B1'].map(recode2)

nesarc_data['USFREQMO'].value_counts(dropna=False)


In [ ]:
# Exercise
# Compare the distributions for:
# S3AQ3B1
# USFREQ
# USFREQMO
# Which variable would be easiest to interpret and why?


## Part 6: Renaming Variables and Simplifying the DataFrame

Cryptic survey variable names make analysis difficult. Professional analysts almost always rename variables so reports and code are easier to understand.


In [ ]:
nesarc_dict = {
 'TAB12MDX':'NICOTINE_DEPENDENCE',
 'CHECK321':'SMOKING_STATUS',
 'S3AQ3B1':'FREQUENCY_OF_SMOKING',
 'S3AQ3C1':'QUANTITY_SMOKED',
 'S2AQ3':'DRANK_ALCOHOL',
 'S2AQ8A':'ALCOHOL_FREQUENCY'
}


In [ ]:
print('Renaming columns...')
nesarc_data.rename(columns=nesarc_dict, inplace=True)
print('Rename complete')


In [ ]:
print('Counts for nicotine dependence')
print(nesarc_data['NICOTINE_DEPENDENCE'].value_counts(sort=True))


Renaming variables improves readability, maintainability and collaboration. Compare NICOTINE_DEPENDENCE to TAB12MDX and consider which is easier to understand when revisiting code several months later.


In [ ]:
# Exercise
# Verify the renaming operation by displaying:
# 1. Column names
# 2. First 5 rows


In [ ]:
nesarc_data = nesarc_data[list(nesarc_dict.values())]

print('New column amount:', len(nesarc_data.columns))


Reducing the dataframe to only the variables required for analysis decreases complexity and improves performance.


In [ ]:
nesarc_data.head()


In [ ]:
nesarc_data.tail()


In [ ]:
nesarc_data.dtypes


In [ ]:
# Exercise
# How many rows and columns are now present?
# Display shape, head and dtypes.


## Reflection Questions

1. Why might a researcher subset data before analysis?
2. Why should invalid survey codes be replaced with NaN?
3. What advantages do descriptive variable names provide?
4. Why is data preparation important before predictive modelling?
5. Which recoding approach (USFREQ or USFREQMO) provides more useful information? Explain your answer.


## Data Dictionary
A data dictionary maps cryptic variable names to meaningful names and supports collaboration.


## Independent Practice
Choose another variable from the codebook. Inspect frequencies, identify invalid values, recode missing data, create a new variable and document all decisions.
